In [ ]:
import os, glob, gc, re, sys
from pathlib import Path
from contextlib import ExitStack

NOTEBOOK_DIR = Path('/home/weiji/restart_exam/code_cleaned/Longrun/date_treatment')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
import numpy as np
import xarray as xr
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

# ============================================================
# Import calculation function
# ============================================================
try:
    from aostools_functions import ComputeEPfluxDiv
except ImportError:
    raise ImportError("Please keep aostools_functions.py in the same directory as this notebook.")

# ============================================================
# Timefixed experiment directories
# ============================================================
ROOT_DIRS = [
    "/mnt/soclim0/public_data/weiji/B2000WCN001002_timefixed",
    "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002_timefixed",
    "/mnt/soclim0/public_data/weiji/B2000WCN007009010011_Clim3D_timefixed",
    "/mnt/soclim0/public_data/weiji/BWCN",
]

# ============================================================
# Settings
# ============================================================
PLEV_STD_PA = np.array([
    10, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000,
    20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000
], dtype=float)

# More complete primitive-equation EP-flux terms:
#   DO_UBAR=True adds the mean-flow shear and absolute-vorticity correction.
#   USE_OMEGA=True passes pressure velocity to aostools so ep2 includes -[u'w'].
DO_UBAR = True
USE_OMEGA = True

# Keep corrected outputs separate from legacy EPflux_daily files.
# Set OUTPUT_SUBDIR_NAME = "EPflux_daily" and OVERWRITE=True only if you deliberately
# want to replace the legacy files.
OUTPUT_VARIANT = "ubar_wcorr" if (DO_UBAR and USE_OMEGA) else ("ubar" if DO_UBAR else ("wcorr" if USE_OMEGA else "qg"))
OUTPUT_SUBDIR_NAME = f"EPflux_daily_{OUTPUT_VARIANT}"

WAVE_ALL = -1
WAVE_1 = 1
WAVE_2 = 2
MAX_WORKERS = 32
OVERWRITE = False

YEAR_RE = re.compile(r"\.cam\.h3\.(\d{4})\.")

print("EP-flux settings:")
print("  DO_UBAR =", DO_UBAR)
print("  USE_OMEGA =", USE_OMEGA)
print("  OUTPUT_SUBDIR_NAME =", OUTPUT_SUBDIR_NAME)
print("  OVERWRITE =", OVERWRITE)

# ============================================================
# Helpers
# ============================================================
def parse_year_from_filename(path: str) -> int:
    m = YEAR_RE.search(os.path.basename(path))
    if not m:
        raise ValueError(f"Cannot parse year from filename: {path}")
    return int(m.group(1))


def detect_prefix(var_dir: str, var_name: str) -> str:
    files = sorted(glob.glob(os.path.join(var_dir, f"*.{var_name}.nc")))
    if not files:
        raise FileNotFoundError(f"No files found in {var_dir} for {var_name}")
    base = os.path.basename(files[0])
    m = re.match(rf"(.+)\.(\d{{4}})\.{re.escape(var_name)}\.nc$", base)
    if not m:
        raise ValueError(f"Cannot detect prefix from filename: {base}")
    return m.group(1)


def file_for(var_dir: str, year_int: int, var_name: str) -> str:
    y = f"{year_int:04d}"
    candidates = glob.glob(os.path.join(var_dir, f"*.cam.h3.{y}.{var_name}.nc"))
    if len(candidates) == 0:
        raise FileNotFoundError(f"Missing {var_name} file for year {y} in {var_dir}")
    if len(candidates) > 1:
        raise RuntimeError(f"Multiple candidates found for {var_name} year {y}: {candidates}")
    return candidates[0]


def compute_pressure_mid(ds: xr.Dataset) -> xr.DataArray:
    return ds["hyam"] * ds["P0"] + ds["hybm"] * ds["PS"]


def interp_profile_logp_4d(v_hyb, p_hyb, p_tgt_pa):
    p_tgt_pa = np.asarray(p_tgt_pa, float)

    def _interp_col(vcol, pcol):
        m = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if m.sum() < 2:
            return np.full(p_tgt_pa.shape, np.nan, float)
        p_use, v_use = pcol[m], vcol[m]
        idx = np.argsort(p_use)
        return np.interp(
            np.log(p_tgt_pa),
            np.log(p_use[idx]),
            v_use[idx],
            left=np.nan,
            right=np.nan
        )

    out = xr.apply_ufunc(
        _interp_col, v_hyb, p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        dask_gufunc_kwargs={"output_sizes": {"plev": len(p_tgt_pa)}},
    )
    return out.assign_coords(plev=("plev", p_tgt_pa))


def get_available_years(U_DIR, V_DIR, T_DIR, OMEGA_DIR=None):
    """Return the common year intersection for required input variables."""
    required = [(U_DIR, "U"), (V_DIR, "V"), (T_DIR, "T")]
    if USE_OMEGA:
        required.append((OMEGA_DIR, "OMEGA"))

    year_sets = []
    for var_dir, var_name in required:
        if var_dir is None:
            raise FileNotFoundError(f"{var_name} directory is required but None was given")
        files = sorted(glob.glob(os.path.join(var_dir, f"*.{var_name}.nc")))
        if not files:
            raise FileNotFoundError(f"No files found for {var_name} in {var_dir}")
        years = {parse_year_from_filename(f) for f in files}
        year_sets.append(years)
    return sorted(set.intersection(*year_sets))


def expected_output_files(OUT_DIR, nyrs):
    return {
        "all_waves": os.path.join(OUT_DIR, "all_waves", f"EPFLUX_all_waves_{nyrs}yr_time_plev_lat.nc"),
        "wave1":     os.path.join(OUT_DIR, "wave1",     f"EPFLUX_wave1_{nyrs}yr_time_plev_lat.nc"),
        "wave2":     os.path.join(OUT_DIR, "wave2",     f"EPFLUX_wave2_{nyrs}yr_time_plev_lat.nc"),
        "wave_rest": os.path.join(OUT_DIR, "wave_rest", f"EPFLUX_wave_rest_{nyrs}yr_time_plev_lat.nc"),
    }


def compute_ep_components(lat_np, u_np, v_np, t_np, w_np, wave):
    return ComputeEPfluxDiv(
        lat=lat_np,
        pres=(PLEV_STD_PA / 100.0),  # hPa
        u=u_np,
        v=v_np,
        t=t_np,
        w=w_np,
        do_ubar=DO_UBAR,
        wave=wave,
    )


def process_one_year(args):
    """
    Read one timefixed year and compute zonal-mean EP flux: time x plev x lat.

    ComputeEPfluxDiv is a wave-mean-flow/zonal-mean diagnostic. It does not
    return a longitude dimension; wave-1/wave-2 are Fourier contributions to
    the zonal-mean co-spectra, not longitude-resolved fluxes.
    """
    year_int, U_DIR, V_DIR, T_DIR, OMEGA_DIR = args

    try:
        fU = file_for(U_DIR, year_int, "U")
        fV = file_for(V_DIR, year_int, "V")
        fT = file_for(T_DIR, year_int, "T")
        fW = file_for(OMEGA_DIR, year_int, "OMEGA") if USE_OMEGA else None

        time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
        with ExitStack() as stack:
            dsU = stack.enter_context(xr.open_dataset(fU, decode_times=time_coder))
            dsV = stack.enter_context(xr.open_dataset(fV, decode_times=time_coder))
            dsT = stack.enter_context(xr.open_dataset(fT, decode_times=time_coder))
            dsW = stack.enter_context(xr.open_dataset(fW, decode_times=time_coder)) if USE_OMEGA else None

            if dsU.sizes.get("time", 0) < 2:
                return f"Too few timesteps for year {year_int:04d}"

            p_mid = compute_pressure_mid(dsU)

            u_std = interp_profile_logp_4d(dsU["U"], p_mid, PLEV_STD_PA)
            v_std = interp_profile_logp_4d(dsV["V"], p_mid, PLEV_STD_PA)
            t_std = interp_profile_logp_4d(dsT["T"], p_mid, PLEV_STD_PA)
            if USE_OMEGA:
                # CAM OMEGA is Pa/s; aostools expects pressure velocity in hPa/s.
                w_std = interp_profile_logp_4d(dsW["OMEGA"] / 100.0, p_mid, PLEV_STD_PA)
            else:
                w_std = None

            u_np = u_std.transpose("time", "plev", "lat", "lon").values
            v_np = v_std.transpose("time", "plev", "lat", "lon").values
            t_np = t_std.transpose("time", "plev", "lat", "lon").values
            w_np = w_std.transpose("time", "plev", "lat", "lon").values if USE_OMEGA else None
            lat_np = dsU["lat"].values

            ep1_all, ep2_all, div1_all, div2_all = compute_ep_components(lat_np, u_np, v_np, t_np, w_np, WAVE_ALL)
            ep1_w1, ep2_w1, div1_w1, div2_w1 = compute_ep_components(lat_np, u_np, v_np, t_np, w_np, WAVE_1)
            ep1_w2, ep2_w2, div1_w2, div2_w2 = compute_ep_components(lat_np, u_np, v_np, t_np, w_np, WAVE_2)

            # Remaining waves = all - wave1 - wave2. This avoids the list-wave
            # branch in aostools_functions.py and keeps the decomposition explicit.
            ep1_rest = ep1_all - ep1_w1 - ep1_w2
            ep2_rest = ep2_all - ep2_w1 - ep2_w2
            div1_rest = div1_all - div1_w1 - div1_w2
            div2_rest = div2_all - div2_w1 - div2_w2

            coords = {
                "time": dsU["time"],
                "plev": PLEV_STD_PA,
                "lat": lat_np,
            }

            ds_out = xr.Dataset(
                data_vars={
                    "ep1_all": (("time", "plev", "lat"), ep1_all),
                    "ep2_all": (("time", "plev", "lat"), ep2_all),
                    "div1_all": (("time", "plev", "lat"), div1_all),
                    "div2_all": (("time", "plev", "lat"), div2_all),

                    "ep1_wave1": (("time", "plev", "lat"), ep1_w1),
                    "ep2_wave1": (("time", "plev", "lat"), ep2_w1),
                    "div1_wave1": (("time", "plev", "lat"), div1_w1),
                    "div2_wave1": (("time", "plev", "lat"), div2_w1),

                    "ep1_wave2": (("time", "plev", "lat"), ep1_w2),
                    "ep2_wave2": (("time", "plev", "lat"), ep2_w2),
                    "div1_wave2": (("time", "plev", "lat"), div1_w2),
                    "div2_wave2": (("time", "plev", "lat"), div2_w2),

                    "ep1_rest": (("time", "plev", "lat"), ep1_rest),
                    "ep2_rest": (("time", "plev", "lat"), ep2_rest),
                    "div1_rest": (("time", "plev", "lat"), div1_rest),
                    "div2_rest": (("time", "plev", "lat"), div2_rest),
                },
                coords=coords,
            )
            ds_out["plev"].attrs.update({"units": "Pa", "positive": "down", "long_name": "pressure"})
            ds_out["lat"].attrs.update(dsU["lat"].attrs)

        gc.collect()
        return ds_out

    except Exception as e:
        return f"Error in year {year_int:04d}: {type(e).__name__}: {str(e)}"


def method_attrs(root_dir, prefix_u, wave_description):
    return {
        "description": f"EP flux full field (time, plev, lat), {wave_description}",
        "root_dir": root_dir,
        "file_prefix": prefix_u,
        "output_variant": OUTPUT_VARIANT,
        "do_ubar": str(DO_UBAR),
        "use_omega_w_correction": str(USE_OMEGA),
        "omega_input_units": "Pa/s",
        "omega_units_passed_to_aostools": "hPa/s",
        "units_ep1": "m2/s2",
        "units_ep2": "hPa*m/s2",
        "units_div1": "m/s/day",
        "units_div2": "m/s/day",
        "zonal_mean_note": "EP flux is a zonal-mean wave-mean-flow diagnostic; no longitude dimension is produced.",
        "history": (
            "Computed from *_timefixed files; no additional internal time shift; "
            f"do_ubar={DO_UBAR}; use_omega_w_correction={USE_OMEGA}; {wave_description}"
        ),
    }


def split_and_write_outputs(ds_full, out_files, root_dir, prefix_u):
    selections = {
        "all_waves": (
            ["ep1_all", "ep2_all", "div1_all", "div2_all"],
            {"ep1_all": "ep1", "ep2_all": "ep2", "div1_all": "div1", "div2_all": "div2"},
            "all zonal waves included",
        ),
        "wave1": (
            ["ep1_wave1", "ep2_wave1", "div1_wave1", "div2_wave1"],
            {"ep1_wave1": "ep1", "ep2_wave1": "ep2", "div1_wave1": "div1", "div2_wave1": "div2"},
            "wave=1",
        ),
        "wave2": (
            ["ep1_wave2", "ep2_wave2", "div1_wave2", "div2_wave2"],
            {"ep1_wave2": "ep1", "ep2_wave2": "ep2", "div1_wave2": "div1", "div2_wave2": "div2"},
            "wave=2",
        ),
        "wave_rest": (
            ["ep1_rest", "ep2_rest", "div1_rest", "div2_rest"],
            {"ep1_rest": "ep1", "ep2_rest": "ep2", "div1_rest": "div1", "div2_rest": "div2"},
            "remaining waves = all - wave1 - wave2",
        ),
    }

    for key, (vars_in, rename_map, wave_description) in selections.items():
        ds = ds_full[vars_in].rename(rename_map)
        ds.attrs = method_attrs(root_dir, prefix_u, wave_description)
        os.makedirs(os.path.dirname(out_files[key]), exist_ok=True)
        print(f"Writing: {out_files[key]}")
        ds.to_netcdf(out_files[key])


def run_one_root(ROOT_DIR):
    U_DIR = os.path.join(ROOT_DIR, "U")
    V_DIR = os.path.join(ROOT_DIR, "V")
    T_DIR = os.path.join(ROOT_DIR, "T")
    OMEGA_DIR = os.path.join(ROOT_DIR, "OMEGA")
    OUT_DIR = os.path.join(ROOT_DIR, OUTPUT_SUBDIR_NAME)
    os.makedirs(OUT_DIR, exist_ok=True)

    years = get_available_years(U_DIR, V_DIR, T_DIR, OMEGA_DIR)
    if not years:
        print(f"No common years found under {ROOT_DIR}")
        return

    nyrs = len(years)
    out_files = expected_output_files(OUT_DIR, nyrs)

    existing = [fp for fp in out_files.values() if os.path.exists(fp) and os.path.getsize(fp) > 0]
    if existing and not OVERWRITE:
        print("\nDetected existing EPflux output, skip run:")
        print(f"ROOT_DIR = {ROOT_DIR}")
        for fp in existing:
            print("   ", fp)
        return

    prefix_u = detect_prefix(U_DIR, "U")

    print("\nEPflux full-field from timefixed files")
    print(f"ROOT_DIR   = {ROOT_DIR}")
    print(f"Prefix     = {prefix_u}")
    print(f"Years      = {years[0]}-{years[-1]}  (n={nyrs})")
    print(f"Workers    = {MAX_WORKERS}")
    print(f"DO_UBAR    = {DO_UBAR}")
    print(f"USE_OMEGA  = {USE_OMEGA}")
    print("Outputs:")
    for k, v in out_files.items():
        print(f"   {k}: {v}")

    args_list = [(y, U_DIR, V_DIR, T_DIR, OMEGA_DIR) for y in years]
    pool_results = []

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
        fut = {ex.submit(process_one_year, args): args[0] for args in args_list}
        for f in tqdm(as_completed(fut), total=len(fut), desc=f"EPflux {os.path.basename(ROOT_DIR)}"):
            res = f.result()
            if isinstance(res, xr.Dataset):
                pool_results.append(res)
            elif res is not None:
                print(f"Warning: {res}")

    if not pool_results:
        print("No data collected.")
        return

    print("Finalizing: concat + sortby(time) + drop duplicate time ...")
    ds_full = xr.concat(pool_results, dim="time").sortby("time")

    tvals = ds_full["time"].values
    _, idx = np.unique(tvals, return_index=True)
    ds_full = ds_full.isel(time=np.sort(idx))

    split_and_write_outputs(ds_full, out_files, ROOT_DIR, prefix_u)
    print("Done!")


def main():
    for root in ROOT_DIRS:
        run_one_root(root)


if __name__ == "__main__":
    main()
